# 03 · Gold: modelo estrella

**Objetivo:** convertir las tablas limpias de `silver` en un **modelo dimensional** listo para Power BI.

**Cómo está pensado el modelo (grano = una línea de pedido):**

```
dim_fecha ---+
dim_cliente -+
dim_producto +--> fact_ventas ---> dim_pedido <--- fact_pagos
dim_vendedor +                          ^
                                        +-------- fact_resenas
```

- **Dimensiones** (`dim_*`): describen *quién, qué, cuándo, dónde*. Una fila por entidad.
- **Hechos** (`fact_*`): contienen las medidas numéricas (precio, flete, puntuación, pago).
- **`dim_pedido`** es el puente: `order_id` **no es único** en `fact_ventas` (un pedido tiene varias líneas), así que pagos y reseñas se relacionan con el pedido y no con la tabla de ventas. Así evitamos duplicar importes en Power BI.

**Regla clave:** nunca mezclar hechos de distinto grano en una sola tabla. Un pedido tiene varios ítems y varios pagos; unirlos multiplicaría los importes.

## 1. `dim_fecha`
No existe en el dataset: la **generamos**. `sequence()` crea un día por fila y `explode()` lo convierte en filas. Añadimos año, trimestre, mes, nombre del mes y día de la semana para segmentar en los reportes.

`fecha_key` (formato `yyyyMMdd`, por ejemplo `20180401`) es una llave numérica corta y rápida para los joins.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_fecha AS
SELECT CAST(date_format(fecha, 'yyyyMMdd') AS INT) AS fecha_key,
fecha,
year(fecha) AS anio,
quarter(fecha) AS trimestre,
month(fecha) AS mes,
date_format(fecha, 'MMMM') AS nombre_mes,
date_format(fecha, 'EEEE') AS dia_semana,
dayofweek(fecha) IN (1, 7) AS es_fin_semana
FROM (
SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) AS fecha
)

## 2. `dim_cliente`
- Se une con `silver.geolocation` por código postal para traer **latitud y longitud** (útil para mapas).
- Es un `LEFT JOIN`: si un código postal no tiene coordenadas, el cliente se conserva igualmente.
- Recuerda: `customer_id` cambia en cada pedido; la **persona real** es `customer_unique_id`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_cliente AS
SELECT c.customer_id,
c.customer_unique_id,
c.customer_city AS ciudad,
c.customer_state AS estado,
c.customer_zip_code_prefix AS codigo_postal,
g.lat,
g.lng
FROM silver.customers c
LEFT JOIN silver.geolocation g
ON c.customer_zip_code_prefix = g.zip_code_prefix

## 3. `dim_producto`
Nos quedamos solo con los atributos útiles para análisis. `categoria` ya viene traducida al inglés desde Silver.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_producto AS
SELECT product_id,
categoria,
product_weight_g AS peso_g,
product_photos_qty AS cantidad_fotos
FROM silver.products

## 4. `dim_vendedor`
Igual que clientes: ciudad y estado del vendedor, más coordenadas por código postal.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_vendedor AS
SELECT s.seller_id,
s.seller_city AS ciudad,
s.seller_state AS estado,
s.seller_zip_code_prefix AS codigo_postal,
g.lat,
g.lng
FROM silver.sellers s
LEFT JOIN silver.geolocation g
ON s.seller_zip_code_prefix = g.zip_code_prefix

## 5. `dim_pedido`
Una fila por pedido (`order_id` único). Es la tabla puente para pagos y reseñas. Incluye el estado y las fechas del ciclo de vida del pedido.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_pedido AS
SELECT order_id,
customer_id,
order_status AS estado_pedido,
order_purchase_timestamp AS fecha_compra,
order_approved_at AS fecha_aprobacion,
order_delivered_carrier_date AS fecha_envio,
order_delivered_customer_date AS fecha_entrega,
order_estimated_delivery_date AS fecha_entrega_estimada
FROM silver.orders

## 6. `fact_ventas` (grano: una línea de pedido)
- `INNER JOIN` con pedidos: los **775 pedidos sin ítems** que vimos en Silver no generan filas aquí, que es lo correcto (no hay venta).
- `total_linea = price + freight_value`.
- `dias_entrega` = días entre compra y entrega. Es `NULL` si el pedido no se entregó.
- `entrega_a_tiempo` es `TRUE` si se entregó en o antes de la fecha estimada, `FALSE` si llegó tarde y `NULL` si aún no se entregó. Ese `NULL` es intencional: un pedido no entregado no cuenta ni como puntual ni como tarde.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.fact_ventas AS
SELECT i.order_id,
i.order_item_id,
o.customer_id,
i.product_id,
i.seller_id,
CAST(date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS fecha_key,
o.order_status AS estado_pedido,
i.price,
i.freight_value,
i.price + i.freight_value AS total_linea,
datediff(o.order_delivered_customer_date, o.order_purchase_timestamp) AS dias_entrega,
CASE WHEN o.order_delivered_customer_date IS NULL THEN NULL
ELSE o.order_delivered_customer_date <= o.order_estimated_delivery_date END AS entrega_a_tiempo
FROM silver.order_items i
INNER JOIN silver.orders o
ON i.order_id = o.order_id

## 7. `fact_pagos` (grano: un pago)
Un pedido puede pagarse en varias cuotas o con varios medios. Se mantiene aparte de `fact_ventas` para **no duplicar importes**.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.fact_pagos AS
SELECT order_id,
payment_sequential AS secuencia_pago,
payment_type AS tipo_pago,
payment_installments AS cuotas,
payment_value AS valor_pago
FROM silver.order_payments

## 8. `fact_resenas` (grano: una reseña)
Puntuación de 1 a 5 y fechas. El texto de los comentarios no se lleva a Gold porque no se usa en el modelo de ventas.

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.fact_resenas AS
SELECT review_id,
order_id,
review_score AS puntuacion,
review_creation_date AS fecha_resena,
review_answer_timestamp AS fecha_respuesta
FROM silver.order_reviews

## 9. Verificación: conteos de filas
`dim_fecha` debe tener 1.096 filas (3 años, 2016 es bisiesto). `fact_ventas` debe coincidir con las líneas de pedido de Silver (~112.650).

In [ ]:
%%sql
SELECT 'dim_fecha' AS tabla, COUNT(*) AS filas FROM gold.dim_fecha
UNION ALL SELECT 'dim_cliente', COUNT(*) FROM gold.dim_cliente
UNION ALL SELECT 'dim_producto', COUNT(*) FROM gold.dim_producto
UNION ALL SELECT 'dim_vendedor', COUNT(*) FROM gold.dim_vendedor
UNION ALL SELECT 'dim_pedido', COUNT(*) FROM gold.dim_pedido
UNION ALL SELECT 'fact_ventas', COUNT(*) FROM gold.fact_ventas
UNION ALL SELECT 'fact_pagos', COUNT(*) FROM gold.fact_pagos
UNION ALL SELECT 'fact_resenas', COUNT(*) FROM gold.fact_resenas

## 10. Verificación: cuadre de importes (la prueba más importante)
Las ventas en Gold deben ser **idénticas** a las de Silver. Si no coinciden, algún join perdió o duplicó filas. La diferencia debe ser 0.

In [ ]:
%%sql
SELECT s.ventas_silver,
g.ventas_gold,
s.ventas_silver - g.ventas_gold AS diferencia
FROM (SELECT SUM(price) AS ventas_silver FROM silver.order_items) s
CROSS JOIN (SELECT SUM(price) AS ventas_gold FROM gold.fact_ventas) g

## 11. Verificación: integridad referencial
Cada fila de `fact_ventas` debe encontrar su cliente, producto, vendedor, fecha y pedido. Todos los valores deben ser 0; si no, Power BI mostrará ventas sin clasificar.

In [ ]:
%%sql
SELECT 'ventas sin cliente' AS regla, COUNT(*) AS filas_con_problema
FROM gold.fact_ventas v LEFT ANTI JOIN gold.dim_cliente c ON v.customer_id = c.customer_id
UNION ALL
SELECT 'ventas sin producto', COUNT(*)
FROM gold.fact_ventas v LEFT ANTI JOIN gold.dim_producto p ON v.product_id = p.product_id
UNION ALL
SELECT 'ventas sin vendedor', COUNT(*)
FROM gold.fact_ventas v LEFT ANTI JOIN gold.dim_vendedor s ON v.seller_id = s.seller_id
UNION ALL
SELECT 'ventas sin fecha', COUNT(*)
FROM gold.fact_ventas v LEFT ANTI JOIN gold.dim_fecha f ON v.fecha_key = f.fecha_key
UNION ALL
SELECT 'ventas sin pedido', COUNT(*)
FROM gold.fact_ventas v LEFT ANTI JOIN gold.dim_pedido d ON v.order_id = d.order_id
UNION ALL
SELECT 'pagos sin pedido', COUNT(*)
FROM gold.fact_pagos p LEFT ANTI JOIN gold.dim_pedido d ON p.order_id = d.order_id
UNION ALL
SELECT 'resenas sin pedido', COUNT(*)
FROM gold.fact_resenas r LEFT ANTI JOIN gold.dim_pedido d ON r.order_id = d.order_id

## 12. Prueba de negocio: top 5 categorías por ventas
Una consulta típica de reporte sobre el modelo estrella: un `JOIN` de la tabla de hechos con una dimensión. Si el modelo está bien, esto funciona sin trucos.

In [ ]:
%%sql
SELECT p.categoria,
ROUND(SUM(v.price), 2) AS ventas,
COUNT(DISTINCT v.order_id) AS pedidos
FROM gold.fact_ventas v
JOIN gold.dim_producto p ON v.product_id = p.product_id
GROUP BY p.categoria
ORDER BY ventas DESC
LIMIT 5